In [5]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get GROQ API key
groq_api_key = os.getenv("GROQ_API_KEY")

# Safety check
if groq_api_key is None:
    raise ValueError("GROQ_API_KEY not found in .env file")

# Optional: set explicitly (not required but safe)
os.environ["GROQ_API_KEY"] = groq_api_key

print("GROQ API key loaded successfully ✅")

GROQ API key loaded successfully ✅


In [6]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)


In [7]:
from langchain_groq import ChatGroq          # Groq LLM
from langchain_core.tools import tool       # Tool decorator
from langchain_core.messages import HumanMessage
import requests

In [8]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [9]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [10]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [11]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)


In [12]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [13]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]

In [14]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [15]:
ai_message = llm_with_tools.invoke(messages)

In [16]:
messages.append(ai_message)

In [17]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'INR', 'target_currency': 'USD'},
  'id': 'vvqhdkbyb',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10, 'conversion_factor': 0.014},
  'id': 'b302fwjvj',
  'type': 'tool_call'}]

In [18]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [19]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'vvqhdkbyb', 'function': {'arguments': '{"base_currency":"INR","target_currency":"USD"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'b302fwjvj', 'function': {'arguments': '{"base_currency_value":10,"conversion_factor":0.014}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 350, 'total_tokens': 397, 'completion_time': 0.077386334, 'completion_tokens_details': None, 'prompt_time': 0.021862905, 'prompt_tokens_details': None, 'queue_time': 0.045009906, 'total_time': 0.099249239}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run-

In [20]:
llm_with_tools.invoke(messages).content

'The conversion factor between INR and USD is 0.01091. Therefore, 10 INR is equivalent to 0.1091 USD.'

In [21]:
# ---------------------------------------------------------------
# 🔥 Groq Instant Tool Calling (Fixed Version)
# ---------------------------------------------------------------

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

load_dotenv()

# ---------------------------------------------------------------
# 🤖 Groq Instant Model
# ---------------------------------------------------------------

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

# ---------------------------------------------------------------
# 🛠️ Single Clean Tool (No Chaining)
# ---------------------------------------------------------------

@tool
def convert_to_meters(value: float, unit: str) -> float:
    """Convert a value from km, cm, or mm into meters."""
    
    conversions = {
        "km": 1000,
        "cm": 0.01,
        "mm": 0.001
    }
    
    factor = conversions.get(unit.lower(), 1)
    return value * factor


# ---------------------------------------------------------------
# 🧠 Bind Tool
# ---------------------------------------------------------------

llm_with_tools = llm.bind_tools([convert_to_meters])

# ---------------------------------------------------------------
# 🚀 User Input
# ---------------------------------------------------------------

messages = [
    HumanMessage(content="Convert 5 km into meters.")
]

response = llm_with_tools.invoke(messages)

# ---------------------------------------------------------------
# 🔁 Tool Execution Loop
# ---------------------------------------------------------------

while response.tool_calls:

    tool_call = response.tool_calls[0]
    result = convert_to_meters.invoke(tool_call["args"])

    messages.append(response)
    messages.append(
        ToolMessage(
            content=str(result),
            tool_call_id=tool_call["id"]
        )
    )

    response = llm_with_tools.invoke(messages)

# ---------------------------------------------------------------
# ✅ Final Answer
# ---------------------------------------------------------------

print("\nFinal Answer:")
print(response.content)


Final Answer:
The result of the function call is 5000.0.


In [23]:
user_query = "Hi how are you?"

messages = [
    HumanMessage(content=user_query)
]

response = llm_with_tools.invoke(messages)

print(response.content)

I'm functioning properly, thanks for asking. What can I help you with today?
